# Clean Qwen3 Dream Fine-Tuning Notebook — Metadata Version

This notebook does the training/export workflow:

1. input/upload the dream dataset at the start;
2. clean the text and **metadata** columns;
3. choose how many cleaned dreams to actually use for training, useful for Free Colab;
4. split each selected dream into **prefix** and **continuation**;
5. train two LoRA/QLoRA variants:
   - **Model A**: sees only the dream beginning;
   - **Model B**: sees the dream beginning + all metadata (`dominant_macro_emotion`, `second_dominant_macro_emotion`, `dream_topic`, `sex`, `age`);
6. export a zip containing the final LoRA adapters.

## 1. Install packages

Run this cell first in Colab. Select a GPU before running: **Runtime → Change runtime type → GPU**.

In [1]:
%pip install -U "transformers>=4.51.0" "accelerate" "datasets" "peft" "bitsandbytes" "scikit-learn" "pandas" "tqdm" "safetensors"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 73.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 72.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 83.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.3/78.3 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.9 MB/s eta 0:00:00
  Attempting uninstall: tqdm
    Found existing installation: tqdm 4.67.3
    Uninstalling tqdm-4.67.3:
      Successfully uninstalled tqdm-4.67.3
  Attempting uninstall: pyarrow
    Found existing installat

## 2. Imports, central dataset input, and configuration

Edit this cell first. In Colab, you can leave `CSV_PATH = None` and upload the CSV immediately when this cell asks.


In [ ]:
import os
import gc
import json
import math
import random
import shutil
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from IPython.display import display
from sklearn.model_selection import KFold, StratifiedKFold

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq,
    set_seed,
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)

try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    files = None
    IN_COLAB = False

os.environ["WANDB_DISABLED"] = "true"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

# ============================================================
# 0. CENTRAL INPUT FILE — EDIT ONLY THIS PART WHEN NEEDED
# ============================================================
# In Colab, leave CSV_PATH as None and upload your dataset when this cell asks.
# Outside Colab, set it explicitly, for example:
# CSV_PATH = "dream_clean_emotion_topic_metadata.csv"
CSV_PATH = None
UPLOAD_DATASET_AT_START = True

MODEL_NAME = "Qwen/Qwen3-1.7B-Base"
OUTPUT_ROOT = Path("qwen3_dream_cv_final_outputs")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
EXPORT_ZIP_NAME = "qwen3_dream_metadata_lora_adapters"

# Column names. Change these only if your dataset uses different names.
TEXT_COL = "dream_text"

# Model B receives all these metadata fields, not only the dominant emotion.
METADATA_COLS = [
    "dominant_macro_emotion",
    "second_dominant_macro_emotion",
    "dream_topic",
    "sex",
    "age",
]

# These columns are used only to keep sampling / CV balanced.
# The prompt still contains ALL columns in METADATA_COLS.
STRATIFY_SAMPLE_COL = "dominant_macro_emotion"
STRATIFY_CV_COL = "dominant_macro_emotion"

# If TEXT_COL is not found, the notebook tries these alternatives.
TEXT_COL_CANDIDATES = ["dream_text", "text_dream", "dream", "text", "report"]

# Dream filtering. Increase MAX_WORDS if your GPU can handle longer sequences.
MIN_WORDS = 30
MAX_WORDS = 350

# Prefix/target split.
SPLIT_RATIO = 0.70

# Tokenization. MAX_TARGET_LENGTH protects the continuation from disappearing under truncation.
MAX_LENGTH = 768
MAX_TARGET_LENGTH = 256

# Cross-validation.
# Free Colab note: leave this False unless you really need CV, because 3 folds × 2 models = 6 trainings.
RUN_CROSS_VALIDATION = False
N_FOLDS = 3
STRATIFY_CV_BY_METADATA = True
SAVE_CV_FOLD_ADAPTERS = False  # Keep False if you only want final deployable adapters.

# Final training on the selected cleaned dataset.
TRAIN_FINAL_MODELS_ON_FULL_DATA = True

# ============================================================
# FREE COLAB TRAINING SUBSET — CHANGE THIS BEFORE RUNNING
# ============================================================
# Choose how many cleaned dreams are used for fine-tuning.
# The notebook will NOT ask for input later.
# Recommended starting values on Free Colab:
#   200-500  = quick test
#   800-1500 = more serious run
#   0 or None = use all cleaned dreams, not recommended on Free Colab
N_DREAMS_FOR_TRAINING = 5000
SAMPLE_STRATIFIED_BY_METADATA = True

# Training hyperparameters.
CV_NUM_TRAIN_EPOCHS = 1
FINAL_NUM_TRAIN_EPOCHS = 1
MAX_STEPS = -1  # -1 means full epochs. Use a small number like 200 only for debugging.
LEARNING_RATE = 1e-4
PER_DEVICE_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
LOGGING_STEPS = 25

# LoRA hyperparameters.
LORA_R = 128
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

AUTO_ZIP_OUTPUTS = True
AUTO_DOWNLOAD_ZIP = True


def upload_dataset_at_start():
    """Upload the CSV once at the beginning, instead of asking later."""
    global CSV_PATH
    if not IN_COLAB or not UPLOAD_DATASET_AT_START:
        return {}
    if CSV_PATH is not None and Path(CSV_PATH).exists():
        return {}

    print("Upload your full dream dataset CSV now.")
    uploaded = files.upload()
    csv_files = [name for name in uploaded.keys() if name.lower().endswith(".csv")]
    if csv_files:
        dream_csvs = [name for name in csv_files if "dream" in name.lower()]
        CSV_PATH = dream_csvs[0] if dream_csvs else csv_files[0]
    return uploaded


UPLOADED_AT_START = upload_dataset_at_start()

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("CSV_PATH:", CSV_PATH)
print("Output root:", OUTPUT_ROOT)
print("Export zip name:", EXPORT_ZIP_NAME + ".zip")


Upload your full dream dataset CSV now.


Saving dream_clean_emotion_topic_metadata.csv to dream_clean_emotion_topic_metadata.csv
GPU available: True
GPU: Tesla T4
CSV_PATH: dream_clean_emotion_topic_metadata.csv
Output root: qwen3_dream_cv_final_outputs
Export zip name: qwen3_dream_metadata_lora_adapters.zip


## 3. Load the full dataset

This cell reuses `CSV_PATH` from the first configuration cell. It only asks for the CSV again if no usable CSV was supplied at the start.


In [3]:
def resolve_csv_path(csv_path=None):
    if csv_path is not None and Path(csv_path).exists():
        return str(csv_path)

    local_csvs = sorted(Path(".").glob("*.csv"))
    if local_csvs:
        dream_csvs = [path for path in local_csvs if "dream" in path.name.lower()]
        chosen = dream_csvs[0] if dream_csvs else local_csvs[0]
        print("Using detected CSV file:", chosen)
        return str(chosen)

    if IN_COLAB:
        print("No CSV found. Upload your full dream dataset CSV now.")
        uploaded = files.upload()
        csv_files = [name for name in uploaded.keys() if name.lower().endswith(".csv")]
        if not csv_files:
            raise FileNotFoundError("No CSV was uploaded.")
        return csv_files[0]

    raise FileNotFoundError(
        "CSV_PATH is None and no CSV was found. Set CSV_PATH to your local CSV path in the first configuration cell."
    )

CSV_PATH = resolve_csv_path(CSV_PATH)
df = pd.read_csv(CSV_PATH)

print("CSV path:", CSV_PATH)
print("Original shape:", df.shape)
print("Columns:")
print(df.columns.tolist())
display(df.head())


CSV path: dream_clean_emotion_topic_metadata.csv
Original shape: (21000, 6)
Columns:
['dream_text', 'dominant_macro_emotion', 'second_dominant_macro_emotion', 'dream_topic', 'sex', 'age']


,dream_text,dominant_macro_emotion,second_dominant_macro_emotion,dream_topic,sex,age
0,"The one at the Meads's house, where it's bigge...",admiration_approval,joy_amusement,topic_02_wheelchair_bonnie_yarn_dovre,female,adult
1,I'm at a family reunion in a large fine house ...,sadness_grief,disappointment_remorse,topic_15_ski_pilot_tornado_clouds,female,adult
2,I watch a plane fly past and shortly realize i...,embarrassment_confusion,disgust_disapproval,topic_15_ski_pilot_tornado_clouds,female,adult
3,Me pulling the green leaves and berries off so...,embarrassment_confusion,admiration_approval,topic_17_kitten_kittens_patio_plant,female,adult
4,I'm in a room that reminds me of (but definite...,embarrassment_confusion,curiosity_surprise,topic_02_wheelchair_bonnie_yarn_dovre,female,adult


## 4. Clean the dataset

This keeps the full dataset except for unusable rows: missing text, missing metadata, too-short dreams, and dreams that are too long for the chosen GPU settings.

In [4]:
def find_text_column(dataframe, preferred_col, candidates):
    if preferred_col in dataframe.columns:
        return preferred_col
    for col in candidates:
        if col in dataframe.columns:
            return col
    raise ValueError(
        f"No text column found. Tried {preferred_col} and candidates {candidates}. "
        f"Available columns: {dataframe.columns.tolist()}"
    )


def validate_metadata_columns(dataframe, metadata_cols):
    missing = [col for col in metadata_cols if col not in dataframe.columns]
    if missing:
        raise ValueError(
            f"Missing metadata columns: {missing}. "
            f"Available columns: {dataframe.columns.tolist()}"
        )


def clean_string_column(series):
    return series.astype(str).str.strip()


TEXT_COL = find_text_column(df, TEXT_COL, TEXT_COL_CANDIDATES)
validate_metadata_columns(df, METADATA_COLS)

df_clean = df.copy()

# Drop rows with missing text or missing metadata before converting to string.
required_cols = [TEXT_COL] + METADATA_COLS
df_clean = df_clean.dropna(subset=required_cols).copy()

df_clean[TEXT_COL] = clean_string_column(df_clean[TEXT_COL])
for col in METADATA_COLS:
    df_clean[col] = clean_string_column(df_clean[col])

invalid_values = {"", "nan", "none", "null"}
df_clean = df_clean[~df_clean[TEXT_COL].str.lower().isin(invalid_values)].copy()
for col in METADATA_COLS:
    df_clean = df_clean[~df_clean[col].str.lower().isin(invalid_values)].copy()

df_clean["n_words"] = df_clean[TEXT_COL].str.split().str.len()
df_clean = df_clean[
    (df_clean["n_words"] >= MIN_WORDS)
    & (df_clean["n_words"] <= MAX_WORDS)
].copy()

df_clean = df_clean.reset_index(drop=True)

print("Text column used:", TEXT_COL)
print("Metadata columns used:", METADATA_COLS)
print("Original rows:", len(df))
print("Clean rows:", len(df_clean))
print("Removed rows:", len(df) - len(df_clean))
print("Word-count summary:")
display(df_clean["n_words"].describe())

print("Metadata distributions:")
for col in METADATA_COLS:
    print(f"\n{col} distribution:")
    display(df_clean[col].value_counts().to_frame("count"))

display(df_clean[[TEXT_COL] + METADATA_COLS + ["n_words"]].head())

Text column used: dream_text
Metadata columns used: ['dominant_macro_emotion', 'second_dominant_macro_emotion', 'dream_topic', 'sex', 'age']
Original rows: 21000
Clean rows: 18093
Removed rows: 2907
Word-count summary:


count    18093.000000
mean       147.147958
std         76.368712
min         50.000000
25%         84.000000
50%        128.000000
75%        195.000000
max        350.000000
Name: n_words, dtype: float64

Metadata distributions:

dominant_macro_emotion distribution:


,count
dominant_macro_emotion,
admiration_approval,5308
embarrassment_confusion,1915
joy_amusement,1773
anger_frustration,1647
fear_anxiety,1585
disappointment_remorse,1493
sadness_grief,1328
curiosity_surprise,1100
optimism_desire,632



second_dominant_macro_emotion distribution:


...


sex distribution:


,count
sex,
female,12506
male,5534
mixed_or_group,53



age distribution:


,count
age,
unknown,4280
longitudinal_multiple_ages,4173
middle_aged,4111
young_adult,2573
adolescent,1129
mixed_or_group,948
adult,404
child,319
older_adult,156


,dream_text,dominant_macro_emotion,second_dominant_macro_emotion,dream_topic,sex,age,n_words
0,"The one at the Meads's house, where it's bigge...",admiration_approval,joy_amusement,topic_02_wheelchair_bonnie_yarn_dovre,female,adult,167
1,I'm at a family reunion in a large fine house ...,sadness_grief,disappointment_remorse,topic_15_ski_pilot_tornado_clouds,female,adult,258
2,I watch a plane fly past and shortly realize i...,embarrassment_confusion,disgust_disapproval,topic_15_ski_pilot_tornado_clouds,female,adult,330
3,Living next door to Loretta in an apartment - ...,embarrassment_confusion,admiration_approval,topic_02_wheelchair_bonnie_yarn_dovre,female,adult,227
4,Kidnapped - I'm on my way somewhere else (by c...,embarrassment_confusion,admiration_approval,topic_03_aliens_enemy_guns_shooting,female,adult,286


## 4b. Choose how many cleaned dreams to use for fine-tuning

This is the Free Colab safety step. The full cleaned dataset can be too large for limited GPU time, so here you can keep only a smaller random/stratified subset. The rest of the notebook uses this selected subset for cross-validation and final fine-tuning.


In [5]:
# Keep a copy of the full cleaned dataset before selecting a smaller training subset.
df_clean_full = df_clean.copy()
N_CLEAN_DREAMS_AVAILABLE = len(df_clean_full)


def _normalize_training_sample_size(value):
    """Return None for all dreams, otherwise return a positive integer sample size."""
    if value is None:
        return None

    try:
        value = int(value)
    except Exception:
        raise ValueError(
            "N_DREAMS_FOR_TRAINING must be an integer, 0, or None. "
            "Example: N_DREAMS_FOR_TRAINING = 500"
        )

    if value <= 0:
        return None
    return value


def stratified_sample_by_column(dataframe, n, stratify_col, seed=42):
    """Sample approximately proportionally by one metadata column."""
    if n is None or n >= len(dataframe):
        return dataframe.copy()

    counts = dataframe[stratify_col].value_counts()

    # If the requested sample is smaller than the number of strata,
    # a normal random sample is safer than forcing one row per stratum.
    if n < len(counts):
        return dataframe.sample(n=n, random_state=seed).copy()

    proportions = counts / counts.sum()
    raw_alloc = proportions * n
    alloc = np.floor(raw_alloc).astype(int)

    # Keep at least one example for each stratum when possible.
    alloc[alloc == 0] = 1

    # If allocation is too large, remove examples from the largest allocated strata.
    while alloc.sum() > n:
        reducible = alloc[alloc > 1]
        if len(reducible) == 0:
            break
        stratum_to_reduce = reducible.idxmax()
        alloc.loc[stratum_to_reduce] -= 1

    # If allocation is too small, add examples according to the largest fractional remainders.
    remainders = (raw_alloc - np.floor(raw_alloc)).sort_values(ascending=False)
    while alloc.sum() < n:
        added = False
        for stratum in remainders.index:
            if alloc.loc[stratum] < counts.loc[stratum]:
                alloc.loc[stratum] += 1
                added = True
                break
        if not added:
            break

    sampled_parts = []
    for stratum, take_n in alloc.items():
        group = dataframe[dataframe[stratify_col] == stratum]
        take_n = min(int(take_n), len(group))
        if take_n > 0:
            sampled_parts.append(group.sample(n=take_n, random_state=seed))

    sampled = pd.concat(sampled_parts, axis=0)

    # Final adjustment in case rounding created a slight mismatch.
    if len(sampled) > n:
        sampled = sampled.sample(n=n, random_state=seed)
    elif len(sampled) < n:
        remaining = dataframe.drop(index=sampled.index)
        add_n = min(n - len(sampled), len(remaining))
        if add_n > 0:
            sampled = pd.concat(
                [sampled, remaining.sample(n=add_n, random_state=seed)],
                axis=0,
            )

    return sampled.sample(frac=1.0, random_state=seed).reset_index(drop=True)


def choose_training_subset_after_cleaning(dataframe):
    global N_DREAMS_FOR_TRAINING

    total = len(dataframe)
    selected_n = _normalize_training_sample_size(N_DREAMS_FOR_TRAINING)

    print("Clean dreams available after filtering:", total)
    print("N_DREAMS_FOR_TRAINING from config cell:", N_DREAMS_FOR_TRAINING)
    print("Tip for Free Colab: start with 300-500; use 800-1500 only if you have enough GPU time.")

    if selected_n is None or selected_n >= total:
        N_DREAMS_FOR_TRAINING = None
        print("Using all cleaned dreams:", total)
        return dataframe.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

    selected_n = int(selected_n)
    N_DREAMS_FOR_TRAINING = selected_n

    if SAMPLE_STRATIFIED_BY_METADATA and STRATIFY_SAMPLE_COL in dataframe.columns:
        subset = stratified_sample_by_column(
            dataframe=dataframe,
            n=selected_n,
            stratify_col=STRATIFY_SAMPLE_COL,
            seed=SEED,
        )
        print(
            f"Using a stratified training subset by {STRATIFY_SAMPLE_COL}: "
            f"{len(subset)} / {total} dreams"
        )
    else:
        subset = dataframe.sample(n=selected_n, random_state=SEED).reset_index(drop=True)
        print(f"Using a random training subset: {len(subset)} / {total} dreams")

    return subset.reset_index(drop=True)


# No interactive input is used here.
# To change the sample size, edit N_DREAMS_FOR_TRAINING in the first configuration cell before running the notebook.
df_clean = choose_training_subset_after_cleaning(df_clean_full)

print("\nRows available after cleaning:", len(df_clean_full))
print("Rows selected for fine-tuning:", len(df_clean))
print(f"Selected {STRATIFY_SAMPLE_COL} distribution:")
display(df_clean[STRATIFY_SAMPLE_COL].value_counts().to_frame("selected_count"))
display(df_clean[[TEXT_COL] + METADATA_COLS + ["n_words"]].head())

Clean dreams available after filtering: 18093
N_DREAMS_FOR_TRAINING from config cell: 10
Tip for Free Colab: start with 300-500; use 800-1500 only if you have enough GPU time.
Using a stratified training subset by dominant_macro_emotion: 10 / 18093 dreams

Rows available after cleaning: 18093
Rows selected for fine-tuning: 10
Selected dominant_macro_emotion distribution:


,selected_count
dominant_macro_emotion,
joy_amusement,3
disappointment_remorse,2
fear_anxiety,2
curiosity_surprise,1
anger_frustration,1
excitement_pride,1


,dream_text,dominant_macro_emotion,second_dominant_macro_emotion,dream_topic,sex,age,n_words
0,Bigger Burger For $1.09 I'm just eaten some f...,joy_amusement,admiration_approval,topic_04_cookies_bread_grocery_store_pan,male,unknown,191
1,I was walking around on campus. To my surpris...,curiosity_surprise,excitement_pride,topic_23_tooth_ward_surgery_patients,female,young_adult,53
2,I remember this dream very vaguely: The surrou...,disappointment_remorse,admiration_approval,topic_20_media_organ_celine_modern_media,female,mixed_or_group,68
3,Father Andrew is on a trip: there is an Indian...,joy_amusement,excitement_pride,topic_21_mass_camden_prayer_pew,female,longitudinal_multiple_ages,70
4,I'm trying to type on a little tiny portable t...,disappointment_remorse,fear_anxiety,topic_16_document_badge_agency_jennifer,female,mixed_or_group,126


## 5. Split each selected dream into prefix and continuation

The model receives the first part of each selected dream and learns to predict the final part. The loss is computed only on the continuation.


In [6]:
def split_dream_report(text, ratio=SPLIT_RATIO):
    words = str(text).split()
    if len(words) < 2:
        return "", ""

    split_idx = int(len(words) * ratio)
    split_idx = max(1, split_idx)
    split_idx = min(split_idx, len(words) - 1)

    prefix = " ".join(words[:split_idx]).strip()
    continuation = " ".join(words[split_idx:]).strip()
    return prefix, continuation

prefixes = []
continuations = []

for dream in df_clean[TEXT_COL].tolist():
    prefix, continuation = split_dream_report(dream)
    prefixes.append(prefix)
    continuations.append(continuation)

df_clean["prefix"] = prefixes
df_clean["continuation"] = continuations

df_clean = df_clean[
    (df_clean["prefix"].str.len() > 0)
    & (df_clean["continuation"].str.len() > 0)
].reset_index(drop=True)

print("Rows after prefix/continuation split:", len(df_clean))
display(df_clean[["prefix", "continuation"] + METADATA_COLS].head())

Rows after prefix/continuation split: 10


,prefix,continuation,dominant_macro_emotion,second_dominant_macro_emotion,dream_topic,sex,age
0,Bigger Burger For $1.09 I'm just eaten some fo...,biting it. I swing my arm away and he is throw...,joy_amusement,admiration_approval,topic_04_cookies_bread_grocery_store_pan,male,unknown
1,I was walking around on campus. To my surprise...,"it was just one, but as I tried to put it back...",curiosity_surprise,excitement_pride,topic_23_tooth_ward_surgery_patients,female,young_adult
2,I remember this dream very vaguely: The surrou...,The problems continued. Then I tried to get th...,disappointment_remorse,admiration_approval,topic_20_media_organ_celine_modern_media,female,mixed_or_group
3,Father Andrew is on a trip: there is an Indian...,"his right hand between an arrow and a tree."" W...",joy_amusement,excitement_pride,topic_21_mass_camden_prayer_pew,female,longitudinal_multiple_ages
4,I'm trying to type on a little tiny portable t...,do about it and I was really frustrated and th...,disappointment_remorse,fear_anxiety,topic_16_document_badge_agency_jennifer,female,mixed_or_group


## 6. Build prompts for Model A and Model B

Model A sees only the beginning. Model B sees the same beginning plus **all metadata**: dominant macro emotion, second dominant macro emotion, topic, sex, and age.

In [7]:
METADATA_LABELS = {
    "dominant_macro_emotion": "dominant_macro_emotion",
    "second_dominant_macro_emotion": "second_dominant_macro_emotion",
    "dream_topic": "dream_topic",
    "sex": "sex",
    "age": "age",
}


def clean_metadata_value(value):
    value = str(value).strip()
    if value == "" or value.lower() in {"nan", "none", "null"}:
        return "unknown"
    return value


def extract_metadata_from_row(row):
    return {col: clean_metadata_value(row[col]) for col in METADATA_COLS}


def format_metadata_block(metadata):
    lines = ["Dream metadata:"]
    for col in METADATA_COLS:
        label = METADATA_LABELS.get(col, col)
        value = clean_metadata_value(metadata.get(col, "unknown"))
        lines.append(f"- {label}: {value}")
    return "\n".join(lines)


def make_prompt(prefix, metadata=None):
    if metadata is None:
        return (
            "Dream beginning:\n"
            f"{prefix}\n\n"
            "Dream continuation:\n"
        )

    return (
        f"{format_metadata_block(metadata)}\n\n"
        "Dream beginning:\n"
        f"{prefix}\n\n"
        "Dream continuation:\n"
    )


def build_prompt_completion_dataframe(dataframe, use_metadata):
    rows = []

    for idx, row in dataframe.iterrows():
        metadata = extract_metadata_from_row(row) if use_metadata else None
        prompt = make_prompt(row["prefix"], metadata=metadata)
        completion = " " + str(row["continuation"]).strip()

        record = {
            "row_id": int(idx),
            "dream_id": row.get("dream_id", idx),
            "uses_metadata": bool(use_metadata),
            "metadata_text": format_metadata_block(extract_metadata_from_row(row)),
            "prompt": prompt,
            "completion": completion,
            "prefix": row["prefix"],
            "target": row["continuation"],
        }
        for col in METADATA_COLS:
            record[col] = row[col]
        rows.append(record)

    return pd.DataFrame(rows)

preview_A = build_prompt_completion_dataframe(df_clean.head(2), use_metadata=False)
preview_B = build_prompt_completion_dataframe(df_clean.head(2), use_metadata=True)

print("Model A prompt example — no metadata:\n")
print(preview_A.loc[0, "prompt"][:800])
print("Model B prompt example — with all metadata:\n")
print(preview_B.loc[0, "prompt"][:1000])

Model A prompt example — no metadata:

Dream beginning:
Bigger Burger For $1.09 I'm just eaten some food from the Greasepit. I hop in my truck. My friend Kevin Simpson is driving. His truck is fixed, and the bumper is on my truck. I'm happy to see him! I ask him if he's hungry: if he wants to eat, and where. I tell him it's my treat: I'll pay. He says the Greasepit! I tell him they've lowered the price of a Bigger Burger from $1.50 back to $1.09, the previous price. I'm on a bicycle at the intersection of Main Street and Last Avenue in Oak Valley near Tom Thumb. A tall blond man is next to me. I swing my arm to hit him in the face with the back of my hand. He catches my hand in his mouth,

Dream continuation:

Model B prompt example — with all metadata:

Dream metadata:
- dominant_macro_emotion: joy_amusement
- second_dominant_macro_emotion: admiration_approval
- dream_topic: topic_04_cookies_bread_grocery_store_pan
- sex: male
- age: unknown

Dream beginning:
Bigger Burger For $1.09 I

## 7. Tokenizer and prompt-masked tokenization

The prompt is context. The continuation is the supervised target. Labels for prompt tokens are set to `-100`, so they do not contribute to the loss.

In [8]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print("EOS token:", tokenizer.eos_token)
print("PAD token:", tokenizer.pad_token)
print("Vocab size:", len(tokenizer))


def tokenize_prompt_completion_batch(batch):
    input_ids_list = []
    attention_mask_list = []
    labels_list = []

    for prompt, completion in zip(batch["prompt"], batch["completion"]):
        prompt_ids = tokenizer(
            prompt,
            add_special_tokens=False,
            truncation=False,
        )["input_ids"]

        completion_ids = tokenizer(
            completion + tokenizer.eos_token,
            add_special_tokens=False,
            truncation=False,
        )["input_ids"]

        # Keep a real supervised target even when the sequence is long.
        completion_ids = completion_ids[:MAX_TARGET_LENGTH]
        available_prompt_len = MAX_LENGTH - len(completion_ids)

        if available_prompt_len < 0:
            completion_ids = completion_ids[:MAX_LENGTH]
            prompt_ids = []
        else:
            # Keep the end of the prompt because it is closest to the continuation.
            prompt_ids = prompt_ids[-available_prompt_len:]

        input_ids = prompt_ids + completion_ids
        attention_mask = [1] * len(input_ids)
        labels = [-100] * len(prompt_ids) + completion_ids.copy()

        input_ids_list.append(input_ids)
        attention_mask_list.append(attention_mask)
        labels_list.append(labels)

    return {
        "input_ids": input_ids_list,
        "attention_mask": attention_mask_list,
        "labels": labels_list,
    }


def tokenize_prompt_dataframe(prompt_df):
    ds = Dataset.from_pandas(
        prompt_df[["prompt", "completion"]].reset_index(drop=True),
        preserve_index=False,
    )
    tokenized = ds.map(
        tokenize_prompt_completion_batch,
        batched=True,
        remove_columns=ds.column_names,
        desc="Tokenizing prompt/completion data",
    )
    return tokenized

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.68k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

EOS token: <|endoftext|>
PAD token: <|endoftext|>
Vocab size: 151669


## 8. Model loading with QLoRA

This loads Qwen3 in 4-bit and trains only LoRA adapter parameters.

In [9]:
def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


def get_compute_dtype():
    if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
        return torch.bfloat16
    return torch.float16


def load_qwen3_qlora_model():
    clear_memory()
    compute_dtype = get_compute_dtype()

    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=compute_dtype,
    )

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=quantization_config,
        device_map="auto",
        torch_dtype=compute_dtype,
        trust_remote_code=True,
    )

    model.config.use_cache = False

    try:
        model = prepare_model_for_kbit_training(
            model,
            use_gradient_checkpointing=True,
        )
    except TypeError:
        model = prepare_model_for_kbit_training(model)
        model.gradient_checkpointing_enable()

    lora_config = LoraConfig(
        task_type="CAUSAL_LM",
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=LORA_TARGET_MODULES,
        bias="none",
    )

    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    return model

## 9. Training utilities

No checkpoints are saved. When `save_adapter=True`, the notebook saves only the final LoRA adapter for that run.

In [14]:
def safe_perplexity(loss):
    try:
        return float(math.exp(loss))
    except OverflowError:
        return float("inf")


def make_training_args(output_dir, num_train_epochs, do_eval):
    return TrainingArguments(
        output_dir=str(output_dir),
        num_train_epochs=num_train_epochs,
        max_steps=MAX_STEPS,
        per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
        per_device_eval_batch_size=PER_DEVICE_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        learning_rate=LEARNING_RATE,
        warmup_ratio=0.03,
        lr_scheduler_type="cosine",
        logging_strategy="steps",
        logging_steps=LOGGING_STEPS,
        eval_strategy="epoch" if do_eval else "no",
        save_strategy="no",
        bf16=(get_compute_dtype() == torch.bfloat16),
        fp16=(get_compute_dtype() == torch.float16),
        optim="paged_adamw_8bit",
        gradient_checkpointing=True,
        report_to="none",
        remove_unused_columns=False,
        dataloader_pin_memory=False,
        load_best_model_at_end=False,
    )


def train_one_adapter(
    experiment_name,
    train_dataset,
    eval_dataset=None,
    num_train_epochs=1,
    output_dir=None,
    save_adapter=False,
):
    if output_dir is None:
        output_dir = OUTPUT_ROOT / experiment_name
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    model = load_qwen3_qlora_model()

    data_collator = DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        model=model,
        padding=True,
        label_pad_token_id=-100,
        return_tensors="pt",
    )

    training_args = make_training_args(
        output_dir=output_dir,
        num_train_epochs=num_train_epochs,
        do_eval=(eval_dataset is not None),
    )

    trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    processing_class=tokenizer,
)

    train_output = trainer.train()

    metrics = {
        "experiment": experiment_name,
        "train_loss": float(train_output.training_loss),
        "num_train_examples": int(len(train_dataset)),
        "num_eval_examples": int(len(eval_dataset)) if eval_dataset is not None else 0,
        "saved_adapter": bool(save_adapter),
        "adapter_dir": str(output_dir) if save_adapter else "",
    }

    if eval_dataset is not None:
        eval_metrics = trainer.evaluate()
        eval_loss = float(eval_metrics["eval_loss"])
        metrics["eval_loss"] = eval_loss
        metrics["eval_perplexity"] = safe_perplexity(eval_loss)
    else:
        metrics["eval_loss"] = np.nan
        metrics["eval_perplexity"] = np.nan

    if save_adapter:
        trainer.save_model(str(output_dir))
        tokenizer.save_pretrained(str(output_dir))

    del trainer
    del model
    clear_memory()

    return metrics

## 10. Cross-validation setup

This now uses only the selected subset from step 4b. On Free Colab, `RUN_CROSS_VALIDATION` is set to `False` by default because cross-validation trains many temporary models.


In [15]:
def build_cv_splits(dataframe):
    if N_FOLDS < 2:
        raise ValueError("N_FOLDS must be at least 2 for cross-validation.")

    if len(dataframe) < N_FOLDS:
        raise ValueError(
            f"The selected training subset has only {len(dataframe)} dreams, "
            f"but N_FOLDS={N_FOLDS}. Reduce N_FOLDS or select more dreams."
        )

    stratify_counts = dataframe[STRATIFY_CV_COL].value_counts()
    can_stratify = (
        STRATIFY_CV_BY_METADATA
        and STRATIFY_CV_COL in dataframe.columns
        and len(stratify_counts) > 1
        and stratify_counts.min() >= N_FOLDS
    )

    if can_stratify:
        print(f"Using StratifiedKFold with {N_FOLDS} folds by {STRATIFY_CV_COL}.")
        splitter = StratifiedKFold(
            n_splits=N_FOLDS,
            shuffle=True,
            random_state=SEED,
        )
        return list(splitter.split(dataframe, dataframe[STRATIFY_CV_COL]))

    print(
        "Using regular KFold because stratification is not possible "
        "with the current metadata distribution."
    )
    splitter = KFold(
        n_splits=N_FOLDS,
        shuffle=True,
        random_state=SEED,
    )
    return list(splitter.split(dataframe))


if RUN_CROSS_VALIDATION:
    cv_splits = build_cv_splits(df_clean)
    print("Number of folds:", len(cv_splits))
else:
    cv_splits = []
    print("RUN_CROSS_VALIDATION is False. Skipping cross-validation setup.")
    print("The final adapters will still be trained on the selected subset.")

RUN_CROSS_VALIDATION is False. Skipping cross-validation setup.
The final adapters will still be trained on the selected subset.


## 11. Run cross-validation for Model A and Model B

This step is expensive. With 3 folds and 2 models, it trains 6 temporary adapters. Model B uses the complete metadata block.

In [16]:
def run_cross_validation():
    cv_results = []
    cv_metrics_path = OUTPUT_ROOT / "cv_metrics.csv"

    variants = [
        ("A_no_metadata", False),
        ("B_with_metadata", True),
    ]

    for fold_idx, (train_idx, val_idx) in enumerate(cv_splits, start=1):
        fold_train_df = df_clean.iloc[train_idx].reset_index(drop=True)
        fold_val_df = df_clean.iloc[val_idx].reset_index(drop=True)

        print("=" * 80)
        print(f"Fold {fold_idx}/{N_FOLDS}")
        print("Train examples:", len(fold_train_df))
        print("Validation examples:", len(fold_val_df))

        for variant_name, use_metadata in variants:
            print("-" * 80)
            print(f"Training {variant_name} on fold {fold_idx}")

            train_prompts = build_prompt_completion_dataframe(
                fold_train_df,
                use_metadata=use_metadata,
            )
            val_prompts = build_prompt_completion_dataframe(
                fold_val_df,
                use_metadata=use_metadata,
            )

            tokenized_train = tokenize_prompt_dataframe(train_prompts)
            tokenized_val = tokenize_prompt_dataframe(val_prompts)

            adapter_dir = OUTPUT_ROOT / "cv_adapters" / f"fold_{fold_idx}_{variant_name}"
            tmp_dir = OUTPUT_ROOT / "tmp_cv_runs" / f"fold_{fold_idx}_{variant_name}"
            output_dir = adapter_dir if SAVE_CV_FOLD_ADAPTERS else tmp_dir

            metrics = train_one_adapter(
                experiment_name=f"fold_{fold_idx}_{variant_name}",
                train_dataset=tokenized_train,
                eval_dataset=tokenized_val,
                num_train_epochs=CV_NUM_TRAIN_EPOCHS,
                output_dir=output_dir,
                save_adapter=SAVE_CV_FOLD_ADAPTERS,
            )

            metrics.update({
                "fold": fold_idx,
                "variant": variant_name,
                "use_metadata": bool(use_metadata),
            })

            cv_results.append(metrics)
            pd.DataFrame(cv_results).to_csv(cv_metrics_path, index=False)
            display(pd.DataFrame(cv_results))

            if not SAVE_CV_FOLD_ADAPTERS:
                shutil.rmtree(tmp_dir, ignore_errors=True)

            del train_prompts, val_prompts, tokenized_train, tokenized_val
            clear_memory()

    cv_results_df = pd.DataFrame(cv_results)
    cv_results_df.to_csv(cv_metrics_path, index=False)

    summary = (
        cv_results_df
        .groupby("variant", as_index=False)
        .agg(
            mean_eval_loss=("eval_loss", "mean"),
            std_eval_loss=("eval_loss", "std"),
            mean_eval_perplexity=("eval_perplexity", "mean"),
            std_eval_perplexity=("eval_perplexity", "std"),
        )
    )
    summary_path = OUTPUT_ROOT / "cv_summary.csv"
    summary.to_csv(summary_path, index=False)

    print("Cross-validation summary:")
    display(summary)
    return cv_results_df, summary

if RUN_CROSS_VALIDATION:
    cv_results_df, cv_summary_df = run_cross_validation()
else:
    print("RUN_CROSS_VALIDATION is False. Skipping cross-validation.")

RUN_CROSS_VALIDATION is False. Skipping cross-validation.


## 12. Train final A and B adapters on the selected subset

These are the adapters you will reuse later. Model A is trained without metadata; Model B is trained with all selected metadata fields.

In [17]:
def save_run_metadata(final_results=None):
    metadata = {
        "created_at": datetime.now().isoformat(),
        "model_name": MODEL_NAME,
        "csv_path": str(CSV_PATH),
        "text_col": TEXT_COL,
        "metadata_cols": METADATA_COLS,
        "stratify_sample_col": STRATIFY_SAMPLE_COL,
        "stratify_cv_col": STRATIFY_CV_COL,
        "clean_rows_before_sampling": int(len(df_clean_full)) if "df_clean_full" in globals() else int(len(df_clean)),
        "training_rows_selected": int(len(df_clean)),
        "n_dreams_for_training": None if N_DREAMS_FOR_TRAINING is None else int(N_DREAMS_FOR_TRAINING),
        "sample_stratified_by_metadata": bool(SAMPLE_STRATIFIED_BY_METADATA),
        "min_words": MIN_WORDS,
        "max_words": MAX_WORDS,
        "split_ratio": SPLIT_RATIO,
        "max_length": MAX_LENGTH,
        "max_target_length": MAX_TARGET_LENGTH,
        "n_folds": N_FOLDS,
        "cv_num_train_epochs": CV_NUM_TRAIN_EPOCHS,
        "final_num_train_epochs": FINAL_NUM_TRAIN_EPOCHS,
        "max_steps": MAX_STEPS,
        "learning_rate": LEARNING_RATE,
        "per_device_batch_size": PER_DEVICE_BATCH_SIZE,
        "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
        "lora_r": LORA_R,
        "lora_alpha": LORA_ALPHA,
        "lora_dropout": LORA_DROPOUT,
        "lora_target_modules": LORA_TARGET_MODULES,
        "final_results": final_results or [],
    }

    metadata_path = OUTPUT_ROOT / "run_metadata.json"
    with open(metadata_path, "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2)

    return metadata_path


def train_final_models_on_full_dataset():
    final_results = []
    variants = [
        ("final_A_no_metadata", False),
        ("final_B_with_metadata", True),
    ]

    for variant_name, use_metadata in variants:
        print("=" * 80)
        print(f"Final training: {variant_name}")
        print("Training examples selected:", len(df_clean))
        if "df_clean_full" in globals():
            print("Clean examples available before selection:", len(df_clean_full))

        prompt_df = build_prompt_completion_dataframe(
            df_clean,
            use_metadata=use_metadata,
        )
        tokenized_full = tokenize_prompt_dataframe(prompt_df)

        adapter_dir = OUTPUT_ROOT / "final_adapters" / variant_name

        metrics = train_one_adapter(
            experiment_name=variant_name,
            train_dataset=tokenized_full,
            eval_dataset=None,
            num_train_epochs=FINAL_NUM_TRAIN_EPOCHS,
            output_dir=adapter_dir,
            save_adapter=True,
        )

        metrics.update({
            "variant": variant_name,
            "use_metadata": bool(use_metadata),
        })

        final_results.append(metrics)
        pd.DataFrame(final_results).to_csv(
            OUTPUT_ROOT / "final_training_results.csv",
            index=False,
        )
        display(pd.DataFrame(final_results))

        del prompt_df, tokenized_full
        clear_memory()

    metadata_path = save_run_metadata(final_results=final_results)
    print("Saved metadata to:", metadata_path)
    return pd.DataFrame(final_results)

if TRAIN_FINAL_MODELS_ON_FULL_DATA:
    final_results_df = train_final_models_on_full_dataset()
else:
    save_run_metadata(final_results=[])
    print("TRAIN_FINAL_MODELS_ON_FULL_DATA is False. Skipping final training.")

Final training: final_A_no_metadata
Training examples selected: 10
Clean examples available before selection: 18093


Tokenizing prompt/completion data:   0%|          | 0/10 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


trainable params: 17,432,576 || all params: 1,738,007,552 || trainable%: 1.0030


...

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


trainable params: 17,432,576 || all params: 1,738,007,552 || trainable%: 1.0030


Step,Training Loss


,experiment,train_loss,num_train_examples,num_eval_examples,saved_adapter,adapter_dir,eval_loss,eval_perplexity,variant,use_metadata
0,final_A_no_metadata,3.211181,10,0,True,qwen3_dream_cv_final_outputs/final_adapters/fi...,NaN,NaN,final_A_no_metadata,False
1,final_B_with_metadata,3.215693,10,0,True,qwen3_dream_cv_final_outputs/final_adapters/fi...,NaN,NaN,final_B_with_metadata,True


Saved metadata to: qwen3_dream_cv_final_outputs/run_metadata.json


## 13. Check saved adapter files

The final adapters are the files you need later. In another notebook, you will load the base Qwen3 model and then load one of these adapter directories.

In [18]:
def list_final_adapter_files():
    final_root = OUTPUT_ROOT / "final_adapters"
    if not final_root.exists():
        print("No final_adapters directory found yet.")
        return

    for adapter_dir in sorted(final_root.iterdir()):
        if adapter_dir.is_dir():
            print("\n", adapter_dir)
            for path in sorted(adapter_dir.iterdir()):
                print("  ", path.name)

list_final_adapter_files()


 qwen3_dream_cv_final_outputs/final_adapters/final_A_no_metadata
   README.md
   adapter_config.json
   adapter_model.safetensors
   chat_template.jinja
   tokenizer.json
   tokenizer_config.json
   training_args.bin

 qwen3_dream_cv_final_outputs/final_adapters/final_B_with_metadata
   README.md
   adapter_config.json
   adapter_model.safetensors
   chat_template.jinja
   tokenizer.json
   tokenizer_config.json
   training_args.bin


## 14. Zip and download outputs

The zip contains:

- `final_adapters/final_A_no_metadata/`
- `final_adapters/final_B_with_metadata/`
- cross-validation metrics;
- final training metadata.

It does **not** contain checkpoints, because checkpoint saving is disabled.

In [19]:
if AUTO_ZIP_OUTPUTS:
    zip_base_name = EXPORT_ZIP_NAME
    zip_path = shutil.make_archive(zip_base_name, "zip", str(OUTPUT_ROOT))
    print("Created zip:", zip_path)
    print("This zip is compatible with the four-model evaluation/generation notebook.")
    print("It contains the LoRA adapter folders expected there:")
    print("- final_adapters/final_A_no_metadata")
    print("- final_adapters/final_B_with_metadata")

    if IN_COLAB and AUTO_DOWNLOAD_ZIP:
        files.download(zip_path)
else:
    print("AUTO_ZIP_OUTPUTS is False. Skipping zip creation.")


Created zip: /content/qwen3_dream_metadata_lora_adapters.zip
This zip is compatible with the four-model evaluation/generation notebook.
It contains the LoRA adapter folders expected there:
- final_adapters/final_A_no_metadata
- final_adapters/final_B_with_metadata


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 15. Minimal reload code for a future notebook

Copy this into your future inference notebook. Choose either the A adapter or the B adapter.

In [20]:
# Minimal reload example for a separate inference notebook.
# This cell is only a reference. You do not need to run it during fine-tuning.

print("Copy this code into your future inference notebook, after unzipping the output folder:")
print(r'''
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

MODEL_NAME = "Qwen/Qwen3-1.7B-Base"
ADAPTER_DIR = "qwen3_dream_cv_final_outputs/final_adapters/final_B_with_metadata"

compute_dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=compute_dtype,
    trust_remote_code=True,
)

model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.eval()
''')

Copy this code into your future inference notebook, after unzipping the output folder:

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

MODEL_NAME = "Qwen/Qwen3-1.7B-Base"
ADAPTER_DIR = "qwen3_dream_cv_final_outputs/final_adapters/final_B_with_metadata"

compute_dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=compute_dtype,
    trust_remote_code=True,
)

model = PeftModel.from_p